In [23]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Import Libraries

In [46]:
import os
from pathlib import Path

import torch
import pandas as pd
import gensim.downloader as api
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_sequence
from numpy import asarray
from torch.nn import CrossEntropyLoss
from sklearn.metrics import classification_report
from torch.optim import Adam
from tqdm.notebook import tqdm


from dataset import *
from model import *
#from trainer import Trainer

import spacy
import nltk


torch.manual_seed(42)

# Read Data

In [25]:
class Trainer:
    def __init__(self, config: Dict):
        """
        Fits end evaluates given model with Adam optimizer.
         Hyperparameters are specified in `config`
        Possible keys are:
            - n_epochs: number of epochs to train
            - lr: optimizer learning rate
            - weight_decay: l2 regularization weight
            - device: on which device to perform training ("cpu" or "cuda")
            - verbose: whether to print anything during training
        :param config: configuration for `Trainer`
        """
        self.config = config
        self.n_epochs = config["n_epochs"]
        self.setup_opt_fn = lambda model: Adam(
            model.parameters(), config["lr"], weight_decay=config["weight_decay"]
        )
        self.model = None
        self.opt = None
        self.history = None
        self.loss_fn = CrossEntropyLoss()
        self.device = config["device"]
        self.verbose = config.get("verbose", True)

    def fit(self, model, train_loader, val_loader):
        """
        Fits model on training data, each epoch evaluates on validation data
        :param model: PyTorch model for toxic comments classification (for example, `RecurrentClassifier`)
        :param train_loader: DataLoader for training data
        :param val_loader: DataLoader for validation data
        :return:
        """
        self.model = model.to(self.device)
        self.opt = self.setup_opt_fn(self.model)
        self.history = {"train_loss": [], "val_loss": [], "val_acc": []}
        for epoch in range(self.n_epochs):
            print(f"Epoch {epoch + 1}/{self.n_epochs}")
            train_info = self._train_epoch(train_loader)
            val_info = self._val_epoch(val_loader)
            self.history["train_loss"].extend(train_info["train_loss"])
            self.history["val_loss"].append(val_info["loss"])
            self.history["val_acc"].append(val_info["acc"])
            print("Val loss ",val_info["loss"])
        return self.model.eval()

    def _train_epoch(self, train_loader):
        self.model.train()
        losses = []
        if self.verbose:
            train_loader = tqdm(train_loader)
        for batch in train_loader:
            self.model.zero_grad()
            texts, labels = batch
            logits = self.model.forward(texts.to(self.device))
            loss = self.loss_fn(logits, labels.to(self.device))
            loss.backward()
            self.opt.step()
            loss_val = loss.item()
            if self.verbose:
                train_loader.set_description(f"Loss={loss_val:.3}")
            losses.append(loss_val)
        return {"train_loss": losses}

    def _val_epoch(self, val_loader):
        self.model.eval()
        all_logits = []
        all_labels = []
        if self.verbose:
            val_loader = tqdm(val_loader)
        with torch.no_grad():
            for batch in val_loader:
                texts, labels = batch
                logits = self.model.forward(texts.to(self.device))
                all_logits.append(logits)
                all_labels.append(labels)
        all_labels = torch.cat(all_labels).to(self.device)
        all_logits = torch.cat(all_logits)
        loss = CrossEntropyLoss()(all_logits, all_labels).item()
        acc = (all_logits.argmax(1) == all_labels).float().mean().item()
        if self.verbose:
            val_loader.set_description(f"Loss={loss:.3}; Acc:{acc:.3}")
        return {"acc": acc, "loss": loss}

    def predict(self, test_loader):
        if self.model is None:
            raise RuntimeError("You should train the model first")
        self.model.eval()
        predictions = []
        with torch.no_grad():
            for batch in test_loader:
                texts, labels = batch
                logits = self.model.forward(texts.to(self.device))
                predictions.extend(logits.argmax(1).tolist())
        return asarray(predictions)

    def save(self, path: str):
        if self.model is None:
            raise RuntimeError("You should train the model first")
        checkpoint = {
            "config": self.model.config,
            "trainer_config": self.config,
            "vocab": self.model.vocab,
            "emb_matrix": self.model.emb_matrix,
            "state_dict": self.model.state_dict(),
        }
        torch.save(checkpoint, path)

    @classmethod
    def load(cls, path: str):
        ckpt = torch.load(path)
        keys = ["config", "trainer_config",
                "vocab", "emb_matrix", "state_dict"]
        for key in keys:
            if key not in ckpt:
                raise RuntimeError(f"Missing key {key} in checkpoint")
        new_model = RecurrentClassifier(
            ckpt["config"], ckpt["vocab"], ckpt["emb_matrix"]
        )
        new_model.load_state_dict(ckpt["state_dict"])
        new_trainer = cls(ckpt["trainer_config"])
        new_trainer.model = new_model
        new_trainer.model.to(new_trainer.device)
        return new_trainer

In [26]:
path = "../../"
train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))

train.head()

,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [27]:
nlp = spacy.load("ru_core_news_sm")
stopwords = nlp.Defaults.stop_words
print(f'Spacy ru stopwords size: {len(stopwords)}', end='\n\n')
' '.join(stopwords)

Spacy ru stopwords size: 768



'наса немного рано такова щ одной иногда каждые нужный х свой чём мною такою иным притом разве таков кстати бишь л там оно покамест казались эй об этот моей й поприще немногим подобного кой мало подобная едва всеми имело скорее едят наши некуда данному поистине вишь чего уже ей явных нынешнее моим моих сперва похожем поэтому самому моему одни подобных нипочем про чему хе другими той довольно отнелижа этой очевидно кем вот потому после ничему мочь самый особые бывала данного через его давно могу дану ешь наверняка ним стать этим такой некому многому только дальше вами ниоткуда чхать сама бац иначе этакий саму впрямь из никогда комья твой что ую п буду того было многое явно ее оне никому сквозь совсем таки эка смогут еще или чей особенно эх само одними данное котором им тех данном которого вдруг каждый ничто которому нем проще тотчас туда можешь нашему ура слишком моего внакладе ибо коль хотела которые просто моги нигде оный этак ж есть дополнительно отсюда ш на чтоб начале недавно некот

In [28]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/vaa2804/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [29]:
nltk_stopwords = nltk.corpus.stopwords.words("russian")
print(f'NLTK ru stopwords size: {len(nltk_stopwords)}', end='\n\n')
' '.join(nltk_stopwords)

NLTK ru stopwords size: 151



'и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между'

In [30]:
nltk_stopwords[:10]

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со']

In [31]:
nlp.stopwords = nltk_stopwords[:100]

In [32]:
train['text'] = train['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        not token.is_stop
        and not token.is_punct
        and not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)

# Label encoding

In [33]:
le = LabelEncoder()

train.rate = le.fit_transform(train.rate)
train.head()

,rate,text
0,3,понравиться март собака дойти лесной озеро зко...
1,4,целое магазин устраивать ассортимент позволять...
2,4,открыться ехать
3,2,громко объявить заботиться пенсионер установит...
4,2,тесно вечный сутолока ряд трудный разойтись гр...


In [34]:
train.describe()

,rate
count,48665.000000
mean,3.055666
std,1.273523
min,0.000000
25%,2.000000
50%,4.000000
75%,4.000000
max,4.000000


# Create pretrained tokenizers

In [35]:
tok = Tokenizer()
tok_texts = [tok.tokenize(t) for t in train.text.values]
vocab = Vocab(tok_texts, max_vocab_size=30000)

In [36]:
tok_texts[:5]

[['понравиться',
  'март',
  'собака',
  'дойти',
  'лесной',
  'озеро',
  'зкотропе',
  'собака',
  'набегаться',
  'нагулялись',
  'домик',
  'чистый',
  'цена',
  'соответствовать',
  'рекомендовать',
  'взять',
  'посуда'],
 ['целое',
  'магазин',
  'устраивать',
  'ассортимент',
  'позволять',
  'ходить',
  'магазин',
  'покупать',
  'дом',
  'акция',
  'сроки',
  'годность',
  'соответствовать',
  'кассир',
  'доброжелательный',
  'правда',
  'последний',
  'год',
  'количество',
  'уменьшиться',
  'раз',
  'работать',
  'касса',
  'очередь',
  'магазин',
  'чисто',
  'размораживать',
  'мыть',
  'холодильник',
  'заходить',
  'неделя',
  'какой',
  'ходовой',
  'товаров',
  'курица',
  'творог',
  'т',
  'д'],
 ['открыться', 'ехать'],
 ['громко',
  'объявить',
  'заботиться',
  'пенсионер',
  'установить',
  'час',
  'посещение',
  'магазин',
  'сеть',
  'заразиться',
  'короновирусом',
  'иметь',
  'время',
  'хлеба',
  'морковь',
  'лимон',
  'т',
  'д',
  'объяснять',
  'заво

In [37]:
vocab.__len__()

29023

In [38]:
vocab.itos[:10]

['<PAD>',
 '<UNK>',
 '<EOS>',
 'магазин',
 'хороший',
 'товар',
 'касса',
 'персонал',
 'ассортимент',
 'удобный']

# Splitting Data

In [39]:
train_texts, train_labels, val_texts, val_labels = train_test_split(train)
train_dataset = TextDataset([tok.tokenize(t)
                            for t in train_texts], train_labels, vocab)
val_dataset = TextDataset([tok.tokenize(t)
                          for t in val_texts], val_labels, vocab)

In [40]:
train_texts[:5]

['цена касса вечно соответствовать цена зал внимательный обходить сторона',
 'отвратительный товар полка ассортимент',
 'постоянный мухлеж ценник остальном хороший магнит дикси',
 'маленький тесный магазин полка лежать просрочка касаться мясной продукция',
 'разочарование магазин овощ никакой вялый картофель свежий единственный плюс свежий выпечка свежий']

In [41]:
train_dataset.texts[:2]

[['цена',
  'касса',
  'вечно',
  'соответствовать',
  'цена',
  'зал',
  'внимательный',
  'обходить',
  'сторона'],
 ['отвратительный', 'товар', 'полка', 'ассортимент']]

In [42]:
train_dataset.__getitem__(0)

([10, 6, 159, 69, 10, 44, 136, 784, 168, 2], 0)

# Create embeddings from tokens

In [43]:
os.environ["GENSIM_DATA_DIR"] = str(Path.cwd())
gensim_model = api.load("word2vec-ruscorpora-300")
emb_matrix = prepare_emb_matrix(gensim_model, vocab)

# Init Model and Config

In [48]:
config = {
    "freeze": False,
    "cell_type": "LSTM",
    "cell_dropout": 0.2,
    "num_layers": 2,
    "hidden_size": 128,
    "out_activation": "relu",
    "bidirectional": True,
    "out_dropout": 0.2,
    "out_sizes": [200],
}

trainer_config = {
    "lr": 3e-3,
    "n_epochs": 5,
    "weight_decay": 1e-5,
    "batch_size": 128,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}
clf_model = RecurrentClassifier(config, vocab, emb_matrix)

# Create Dataloaders and Train

In [49]:
train_dataloader = DataLoader(train_dataset,
                              batch_size=trainer_config["batch_size"],
                              shuffle=True,
                              num_workers=0,
                              collate_fn=train_dataset.collate_fn)
val_dataloader = DataLoader(val_dataset,
                            batch_size=trainer_config["batch_size"],
                            shuffle=False,
                            num_workers=0,
                            collate_fn=val_dataset.collate_fn)
t = Trainer(trainer_config)
t.fit(clf_model, train_dataloader, val_dataloader)

Epoch 1/5


  0%|          | 0/324 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

Train loss  [1.8064523935317993, 1.5921828746795654, 1.2226734161376953, 1.4356168508529663, 1.5454312562942505, 1.1617356538772583, 1.2345975637435913, 1.2621084451675415, 1.1606181859970093, 1.0408101081848145, 0.9769644737243652, 0.9937536120414734, 1.0956122875213623, 1.0303277969360352, 0.8861694931983948, 0.9230989813804626, 0.9272538423538208, 0.9017505049705505, 0.9094640612602234, 0.9631466865539551, 0.9183959364891052, 1.116673469543457, 0.9633764028549194, 0.8728729486465454, 0.9447911381721497, 0.8054189085960388, 0.8240413665771484, 0.9646854996681213, 0.9433165192604065, 0.9562667608261108, 0.8600761294364929, 0.8183967471122742, 0.8175795078277588, 0.8398489952087402, 0.8507928848266602, 0.8129183650016785, 0.8836572170257568, 0.8828200101852417, 0.8810104727745056, 0.7335713505744934, 0.81934654712677, 0.835889995098114, 0.9033359885215759, 0.934067964553833, 0.9306219220161438, 0.8887559771537781, 0.9998186826705933, 0.8010432124137878, 0.8229913115501404, 0.7908316254

  0%|          | 0/324 [00:00<?, ?it/s]

...

Train loss  [0.22725339233875275, 0.2518540620803833, 0.3961808681488037, 0.36150094866752625, 0.26526251435279846, 0.4515077769756317, 0.30476894974708557, 0.24636772274971008, 0.32903629541397095, 0.3738667070865631, 0.22628438472747803, 0.2669571042060852, 0.3312516212463379, 0.3120933771133423, 0.34154772758483887, 0.28672119975090027, 0.2731322646141052, 0.44326287508010864, 0.40787988901138306, 0.3117011785507202, 0.40674668550491333, 0.2642173171043396, 0.2920769155025482, 0.19018736481666565, 0.46355175971984863, 0.3411451280117035, 0.4030241072177887, 0.2525096833705902, 0.292729914188385, 0.3874954283237457, 0.2677655816078186, 0.3935098946094513, 0.32567304372787476, 0.33857661485671997, 0.4175581932067871, 0.3313344419002533, 0.36520981788635254, 0.3179677426815033, 0.2705135643482208, 0.31935811042785645, 0.27606305480003357, 0.3480052947998047, 0.30434346199035645, 0.3093259334564209, 0.20529420673847198, 0.24002942442893982, 0.22724592685699463, 0.2261514663696289, 0.209

  0%|          | 0/324 [00:00<?, ?it/s]

  0%|          | 0/58 [00:00<?, ?it/s]

Train loss  [0.2888123691082001, 0.2609193027019501, 0.2513764798641205, 0.28204935789108276, 0.2758316993713379, 0.20054160058498383, 0.30133283138275146, 0.23529498279094696, 0.32541805505752563, 0.3728162348270416, 0.24444904923439026, 0.2843824326992035, 0.30237895250320435, 0.2139168083667755, 0.19181330502033234, 0.31469008326530457, 0.15198688209056854, 0.2562277317047119, 0.21104398369789124, 0.2909272015094757, 0.27627259492874146, 0.26083672046661377, 0.18386709690093994, 0.3705122768878937, 0.152248352766037, 0.3349694609642029, 0.2508477568626404, 0.22886697947978973, 0.3743365705013275, 0.3493349552154541, 0.3577320873737335, 0.23790450394153595, 0.30650344491004944, 0.36145520210266113, 0.22603917121887207, 0.26569297909736633, 0.178852379322052, 0.1738128662109375, 0.28645017743110657, 0.33052095770835876, 0.27616140246391296, 0.32162952423095703, 0.22855816781520844, 0.23171420395374298, 0.2665656507015228, 0.24569280445575714, 0.2816125452518463, 0.19548673927783966, 0

RecurrentClassifier(
  (embeddings): Embedding(29023, 300, padding_idx=0)
  (cell): LSTM(300, 128, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (out_dropout): Dropout(p=0.2, inplace=False)
  (out_proj): Sequential(
    (0): Linear(in_features=512, out_features=200, bias=True)
    (1): Linear(in_features=200, out_features=6, bias=True)
  )
)

# Save Model

In [ ]:
t.save("baseline_model.ckpt")

# Load pretrained Model

In [ ]:
t = Trainer.load("baseline_model.ckpt")

# Define predict function

In [ ]:
def predict(model, text):
    tok_text = tok.tokenize(text)
    indexed_text = torch.tensor(vocab.vectorize(tok_text)).to(t.device)
    genre = model(pack_sequence([indexed_text])).argmax().item()
    return genre

In [50]:
valid_predictions = t.predict(val_dataloader)
print(classification_report(val_labels, valid_predictions))

NameError: name 'classification_report' is not defined

# Get testset predictions

In [ ]:
test['text'] = test['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        not token.is_stop
        and not token.is_punct
        and not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)

In [ ]:
test_dataloader = DataLoader(TextDataset([tok.tokenize(t) for t in test.text.values], [-1] * test.shape[0], vocab),
                             batch_size=trainer_config["batch_size"],
                             shuffle=False,
                             num_workers=0,
                             collate_fn=val_dataset.collate_fn)

predictions = t.predict(test_dataloader)

# Create submission

In [ ]:
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))
sample_submission["rate"] = predictions
sample_submission.rate = le.inverse_transform(sample_submission.rate)
sample_submission.head()

In [ ]:
sample_submission.to_csv("submission.csv", index=False)